# Kimi K3 autoregressive inference

This notebook rebuilds the architecture from YAML, loads a trained checkpoint and generates with one `prefill` followed by cached `decode_step` calls using the native KDA/MLA state. The repository does not ship a trained checkpoint yet, so all cells are intentionally unexecuted.

In [ ]:
from pathlib import Path

from configuration import resolve_kimi_pipeline_profile
from data import load_tokenizer_from_data_yaml
from inference import (
    ModelLoadConfig,
    inference_autoregressive,
    load_generation_config,
    load_kimi_checkpoint,
)

In [ ]:
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
# Change only PROFILE_NAME and CHECKPOINT to switch inference experiments.
PROFILE_NAME = 't4_wikitext'
# PROFILE_NAME = 'cpu_smoke'
# PROFILE_NAME = 'low_gpu'
# PROFILE_NAME = 't4_retrieval'
# PROFILE_NAME = 'gpu_24gb'
# PROFILE_NAME = 'gpu_48gb'
# PROFILE_NAME = 'gpu_80gb'
# PROFILE_NAME = 'canonical'  # Metadata only until distributed loading exists.

PROFILE = resolve_kimi_pipeline_profile(
    ROOT / 'config/kimi_full_pipeline' / PROFILE_NAME
)
DATA_YAML = PROFILE.data
MODEL_YAML = PROFILE.model
INFERENCE_YAML = ROOT / 'config/inference/greedy.yaml'
CHECKPOINT = ROOT / 'checkpoints/t4_wikitext_213m/kimi_k3_t4_wikitext_213m_epoch_0002.pt'
# CHECKPOINT = ROOT / 'checkpoints/t4_retrieval_246m/kimi_k3_t4_retrieval_246m_epoch_0002.pt'
# CHECKPOINT = ROOT / 'checkpoints/gpu_48gb_pcc/kimi_k3_gpu_48gb_pcc_epoch_0002.pt'

## 1. Tokenizer and checkpoint

The tokenizer is reconstructed from the data YAML. Hugging Face profiles require the tokenizer file cached during data preparation.

In [ ]:
tokenizer = load_tokenizer_from_data_yaml(DATA_YAML)
loaded = load_kimi_checkpoint(
    MODEL_YAML,
    CHECKPOINT,
    tokenizer=tokenizer,
    load_config=ModelLoadConfig(device='cuda', precision='fp16'),
)
model = loaded.model

## 2. Generation

The master function tokenizes the prompt, performs one prefill, decodes one cached token at a time and converts the result back to text.

In [ ]:
generation_config = load_generation_config(INFERENCE_YAML)
output = inference_autoregressive(
    model,
    'Once upon a time',
    tokenizer=tokenizer,
    generation_config=generation_config,
)
print(output.completion_text)

In [ ]:
{
    'finish_reason': output.finish_reason,
    'generated_tokens': output.generated_tokens,
    'tokens_per_second': output.tokens_per_second,
    'cache': output.cache_stats,
}